# Human Activity Recognition Using Smartphone Sensor Data
## Exploratory Data Analysis, Statistical Visualization, and Deep Learning

**Student:** [Your Name]  
**Roll No.:** [Your Roll Number]  
**Department:** CSE (AI&ML)  
**Platform:** Google Colab  
**Dataset:** UCI Human Activity Recognition Using Smartphones

### Project objective
Explore smartphone accelerometer and gyroscope feature measurements, identify statistical patterns, and train a Multilayer Perceptron (MLP) to classify six activities. This notebook computes all reported metrics from the downloaded data; no performance result is assumed in advance.

### Seven-stage visualization workflow used in this project
1. **Problem framing:** define the activity-recognition question and target label.
2. **Data acquisition:** download the public UCI dataset.
3. **Data preparation:** inspect types, missing values, duplicates, and feature scaling.
4. **Exploratory analysis:** summarize distributions and relationships.
5. **Visual encoding:** select plots that match each analytical question.
6. **Interpretation and communication:** explain what each plot shows and why it matters.
7. **Evaluation and refinement:** evaluate the model and note limitations or follow-up analyses.

> Run the notebook from top to bottom in Google Colab. The dataset is downloaded by the notebook. The final section prints the actual results to use in the report.

## 1. Setup and imports
The notebook uses common Python data-science libraries and TensorFlow/Keras. If Colab requests a runtime restart after a package update, restart and run from the top.

In [ ]:
import os, zipfile, urllib.request, warnings, json, random, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    roc_curve, auc, roc_auc_score
)
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

## 2. Download and load the dataset
The official dataset provides separate training and test partitions, 561 engineered sensor features, subject identifiers, and six activity labels. We preserve the official test partition and split only the original training partition into train and validation sets.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ZIP_PATH = Path("har_smartphones.zip")
DATA_DIR = Path("UCI HAR Dataset")

if not DATA_DIR.exists():
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(".")
else:
    print("Dataset directory already exists; skipping download.")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "UCI HAR Dataset folder not found. Check the UCI download URL and extract the archive."
    )

def read_matrix(path):
    return pd.read_csv(path, sep=r"\s+", header=None)

features = pd.read_csv(DATA_DIR / "features.txt", sep=r"\s+", header=None, names=["index", "feature"])
activity_table = pd.read_csv(DATA_DIR / "activity_labels.txt", sep=r"\s+", header=None, names=["activity_id", "activity"])

X_train_original = read_matrix(DATA_DIR / "train" / "X_train.txt")
y_train_original = read_matrix(DATA_DIR / "train" / "y_train.txt").iloc[:, 0]
subjects_train_original = read_matrix(DATA_DIR / "train" / "subject_train.txt").iloc[:, 0]

X_test = read_matrix(DATA_DIR / "test" / "X_test.txt")
y_test = read_matrix(DATA_DIR / "test" / "y_test.txt").iloc[:, 0]
subjects_test = read_matrix(DATA_DIR / "test" / "subject_test.txt").iloc[:, 0]

feature_names = features["feature"].astype(str).tolist()
X_train_original.columns = feature_names
X_test.columns = feature_names

label_map = dict(zip(activity_table["activity_id"], activity_table["activity"]))
y_train_original = y_train_original.map(label_map)
y_test = y_test.map(label_map)

print("Training partition:", X_train_original.shape)
print("Test partition:", X_test.shape)
print("Number of features:", len(feature_names))
print("Activity labels:", sorted(y_train_original.unique()))
display(activity_table)
display(X_train_original.head())

## 3. Data understanding and quality checks
We check dataset shape, data types, missing values, exact duplicate rows, and descriptive statistics. The UCI files are numeric engineered features, so the feature matrix should be numeric. A zero missing-value count is still reported rather than assumed.

In [ ]:
print("Training features shape:", X_train_original.shape)
print("Test features shape:", X_test.shape)
print("\nData types:")
display(X_train_original.dtypes.value_counts().rename_axis("dtype").to_frame("column_count"))

missing_by_column = X_train_original.isna().sum()
print("Total missing feature cells:", int(missing_by_column.sum()))
print("Columns with missing values:", int((missing_by_column > 0).sum()))
print("Exact duplicate feature rows in original training partition:", int(X_train_original.duplicated().sum()))

print("\nDescriptive statistics (first 12 features):")
display(X_train_original.iloc[:, :12].describe().T)

print("\nTraining activity counts:")
display(y_train_original.value_counts().rename_axis("activity").to_frame("count"))

## 4. Preprocessing and split strategy
- We first create a validation split from the original training partition using stratification.
- The provided test partition is kept untouched until final evaluation.
- Missing values, if any, are imputed using training-set medians only.
- Features are standardized using training-set statistics only to prevent leakage.
- Exact duplicate rows are counted and reported; rows are not automatically deleted because repeated sensor feature vectors can be legitimate observations.
- Outliers are inspected using boxplots and IQR counts. They are not automatically removed because unusual sensor measurements may be meaningful.

In [ ]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_original, y_train_original,
    test_size=0.20, random_state=SEED, stratify=y_train_original
)

# Use training-only medians for any missing-value imputation.
train_medians = X_train_raw.median(numeric_only=True)
X_train_imp = X_train_raw.fillna(train_medians)
X_val_imp = X_val_raw.fillna(train_medians)
X_test_imp = X_test.fillna(train_medians)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_imp)
X_val = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

class_names = sorted(y_train_original.unique())
class_to_int = {name: idx for idx, name in enumerate(class_names)}
y_train_int = y_train.map(class_to_int).to_numpy()
y_val_int = y_val.map(class_to_int).to_numpy()
y_test_int = y_test.map(class_to_int).to_numpy()

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test_scaled.shape)
print("Training class balance:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 5. Exploratory data analysis and visualizations
Each plot includes the question it addresses and an interpretation guide. After running the cells, use the displayed pattern and numerical summaries to write the data-specific finding in your report; do not claim a pattern that the chart does not show.

In [ ]:
# Visualization 1: activity class distribution
fig, ax = plt.subplots(figsize=(9, 5))
order = y_train_original.value_counts().index
sns.countplot(x=y_train_original, order=order, ax=ax)
ax.set_title("Figure 1. Activity Class Distribution in Training Data")
ax.set_xlabel("Activity")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_activity_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: number of training records in each activity class.")
print("Why created: to check class balance before model training.")
print("Interpretation: compare bar heights. A visibly smaller class may require class-wise metric attention.")

In [ ]:
# Visualization 2: distributions of selected continuous features
selected_features = [
    "tBodyAcc-mean()-X", "tBodyAcc-mean()-Y", "tBodyAcc-mean()-Z",
    "tGravityAcc-mean()-X", "tGravityAcc-mean()-Y", "tGravityAcc-mean()-Z"
]
available_features = [f for f in selected_features if f in X_train_original.columns]
if not available_features:
    available_features = X_train_original.columns[:6].tolist()

plot_df = X_train_original[available_features].melt(var_name="feature", value_name="value")
g = sns.displot(data=plot_df, x="value", col="feature", col_wrap=3, kind="hist",
                bins=35, height=3, facet_kws={"sharex": False, "sharey": False})
g.fig.suptitle("Figure 2. Distributions of Selected Sensor Features", y=1.03)
g.savefig(FIG_DIR / "02_feature_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: frequency distributions of selected sensor-derived features.")
print("Why created: to inspect spread, skewness, concentration, and possible unusual values.")
print("Interpretation: note whether each distribution is symmetric, skewed, concentrated, or multi-peaked.")

In [ ]:
# Visualization 3: boxplots for outlier screening
box_features = available_features[:6]
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=X_train_original[box_features], orient="h", ax=ax)
ax.set_title("Figure 3. Boxplots for Outlier Screening")
ax.set_xlabel("Feature value")
ax.set_ylabel("Sensor feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_outlier_boxplots.png", dpi=160, bbox_inches="tight")
plt.show()

q1 = X_train_original[box_features].quantile(0.25)
q3 = X_train_original[box_features].quantile(0.75)
iqr = q3 - q1
outlier_counts = ((X_train_original[box_features] < (q1 - 1.5 * iqr)) |
                  (X_train_original[box_features] > (q3 + 1.5 * iqr))).sum()
print("IQR-based outlier counts in selected features:")
display(outlier_counts.rename("outlier_count").to_frame())
print("What it shows: median, interquartile range, and points beyond 1.5×IQR whiskers.")
print("Why created: to screen for unusually high or low feature values.")
print("Interpretation: outlier flags are candidates for inspection, not proof of data errors; sensor extremes may be valid.")

In [ ]:
# Visualization 4: correlation heatmap on a manageable subset of features
corr_features = available_features[:6]
corr = X_train_original[corr_features].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Figure 4. Correlation Heatmap of Selected Sensor Features")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: pairwise linear correlation between selected features.")
print("Why created: to explore redundancy and relationships among sensor measurements.")
print("Interpretation: values near +1 indicate strong positive linear association; near -1 indicate negative association; near 0 indicate weak linear association.")

In [ ]:
# Visualization 5: relationship between two features, colored by activity
scatter_x = "tBodyAcc-mean()-X" if "tBodyAcc-mean()-X" in X_train_original.columns else X_train_original.columns[0]
scatter_y = "tBodyAcc-mean()-Y" if "tBodyAcc-mean()-Y" in X_train_original.columns else X_train_original.columns[1]
scatter_sample = X_train_original[[scatter_x, scatter_y]].copy()
scatter_sample["activity"] = y_train_original.values
if len(scatter_sample) > 5000:
    scatter_sample = scatter_sample.sample(5000, random_state=SEED)

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=scatter_sample, x=scatter_x, y=scatter_y, hue="activity",
                alpha=0.55, s=22, ax=ax)
ax.set_title("Figure 5. Relationship Between Two Sensor Features by Activity")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_feature_relationship.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: the relationship between two selected features, with color representing activity.")
print("Why created: to see whether activities occupy different regions in feature space.")
print("Interpretation: clearer separation suggests these features may help classification; overlap suggests more features or nonlinear boundaries are needed.")

In [ ]:
# Visualization 6: skewness of selected features (statistical distribution summary)
skewness = X_train_original[available_features].skew().sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
skewness.plot(kind="barh", ax=ax)
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_title("Figure 6. Skewness of Selected Sensor Features")
ax.set_xlabel("Skewness")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_feature_skewness.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: asymmetry of feature distributions.")
print("Why created: skewness complements histograms with a numerical summary.")
print("Interpretation: positive values indicate right-skew; negative values indicate left-skew; values near zero indicate approximate symmetry.")

### Statistical and probability-distribution analysis
The target is a **discrete categorical variable** (six activities), summarized with a frequency distribution. The engineered sensor features are **continuous numerical variables**, explored using histograms, boxplots, means, medians, standard deviations, and skewness. A normal distribution is not assumed; histograms and skewness are used to assess shape. Pearson correlation is used only as a measure of linear association, not causation.

In [ ]:
# Compact statistical summary to cite in the report
stat_summary = X_train_original[available_features].agg(["mean", "median", "std", "min", "max"]).T
stat_summary["skewness"] = X_train_original[available_features].skew()
display(stat_summary)

activity_counts = y_train_original.value_counts().sort_index()
activity_props = y_train_original.value_counts(normalize=True).sort_index()
activity_summary = pd.DataFrame({"count": activity_counts, "proportion": activity_props})
display(activity_summary)

## 6. Deep learning model: Multilayer Perceptron (MLP)
An MLP is suitable for the 561-dimensional engineered feature vectors. We compare a small set of configurations using validation accuracy, then evaluate the selected model exactly once on the untouched test partition. Dropout and early stopping help reduce overfitting.

In [ ]:
def build_mlp(input_dim, num_classes, units=128, dropout=0.25, learning_rate=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(units, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units // 2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

configs = [
    {"units": 128, "dropout": 0.25, "learning_rate": 1e-3, "batch_size": 64},
    {"units": 256, "dropout": 0.30, "learning_rate": 5e-4, "batch_size": 64},
]
tuning_results = []
trained_candidates = []

for i, cfg in enumerate(configs, start=1):
    print(f"Training configuration {i}/{len(configs)}: {cfg}")
    tf.keras.backend.clear_session()
    candidate = build_mlp(
        X_train.shape[1], len(class_names),
        units=cfg["units"], dropout=cfg["dropout"],
        learning_rate=cfg["learning_rate"]
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=0
        )
    ]
    hist = candidate.fit(
        X_train, y_train_int,
        validation_data=(X_val, y_val_int),
        epochs=60,
        batch_size=cfg["batch_size"],
        callbacks=callbacks,
        verbose=0
    )
    best_val_acc = float(max(hist.history["val_accuracy"]))
    tuning_results.append({**cfg, "best_validation_accuracy": best_val_acc,
                           "epochs_run": len(hist.history["loss"])})
    trained_candidates.append((candidate, hist, cfg))

tuning_df = pd.DataFrame(tuning_results).sort_values("best_validation_accuracy", ascending=False)
display(tuning_df)
best_idx = int(tuning_df.index[0])
# tuning_df retains original indexes from tuning_results
best_model, history, best_config = trained_candidates[best_idx]
print("Selected configuration:", best_config)
print("Selection criterion: highest validation accuracy; test set not used for tuning.")

## 7. Training curves
These plots compare training and validation loss/accuracy over epochs. A widening train-validation gap can indicate overfitting; high loss on both sets can indicate underfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history.history["loss"], label="Training loss")
ax.plot(history.history["val_loss"], label="Validation loss")
ax.set_title("Figure 7. Loss vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Sparse categorical cross-entropy")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "07_loss_vs_epoch.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history.history["accuracy"], label="Training accuracy")
ax.plot(history.history["val_accuracy"], label="Validation accuracy")
ax.set_title("Figure 8. Accuracy vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "08_accuracy_vs_epoch.png", dpi=160, bbox_inches="tight")
plt.show()

print("Interpretation: compare train and validation curves. Use their actual trajectories to describe convergence, overfitting, and whether early stopping appears useful.")

## 8. Final test evaluation
The official test set is used here only after model selection. We report accuracy, class-wise precision/recall/F1, a confusion matrix, and multiclass ROC-AUC where calculable.

In [ ]:
test_probabilities = best_model.predict(X_test_scaled, verbose=0)
y_pred_int = np.argmax(test_probabilities, axis=1)
y_pred = np.array(class_names)[y_pred_int]

test_accuracy = accuracy_score(y_test_int, y_pred_int)
print(f"TEST ACCURACY: {test_accuracy:.4f}")
print("\nClassification report:")
report_dict = classification_report(
    y_test_int, y_pred_int, target_names=class_names, output_dict=True, zero_division=0
)
print(classification_report(
    y_test_int, y_pred_int, target_names=class_names, digits=4, zero_division=0
))

cm = confusion_matrix(y_test_int, y_pred_int)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title("Figure 9. Confusion Matrix on the Official Test Set")
ax.set_xlabel("Predicted activity")
ax.set_ylabel("True activity")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "09_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

print("What it shows: correct classifications on the diagonal and errors off the diagonal.")
print("Why created: overall accuracy can hide which activity pairs are confused.")
print("Interpretation: inspect the largest off-diagonal cells and discuss those activity pairs.")

In [ ]:
# Multiclass ROC curves and AUC (one-vs-rest)
y_test_onehot = label_binarize(y_test_int, classes=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(8, 6))
auc_by_class = {}
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_onehot[:, i], test_probabilities[:, i])
    class_auc = auc(fpr, tpr)
    auc_by_class[name] = float(class_auc)
    ax.plot(fpr, tpr, label=f"{name} (AUC={class_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("Figure 10. One-vs-Rest ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "10_multiclass_roc.png", dpi=160, bbox_inches="tight")
plt.show()

macro_auc = roc_auc_score(y_test_onehot, test_probabilities, average="macro", multi_class="ovr")
print("Macro one-vs-rest ROC-AUC:", round(float(macro_auc), 4))
print("AUC by class:", auc_by_class)
print("Interpretation: curves nearer the upper-left and larger AUC values indicate stronger class-vs-rest discrimination.")

## 9. Feature analysis and permutation importance
A neural network does not provide simple coefficient-based feature importance. We estimate permutation importance on a reproducible sample of the official test set using accuracy decrease after shuffling each feature. This is computationally expensive, so we restrict it to a subset of the most variable features and a small sample. Importance is predictive, not causal.

In [ ]:
# Select a manageable subset of features with highest variance for permutation analysis.
variance_rank = X_train_original.var().sort_values(ascending=False)
importance_features = variance_rank.head(20).index.tolist()
rng = np.random.default_rng(SEED)
sample_n = min(1200, len(X_test_scaled))
sample_idx = rng.choice(len(X_test_scaled), size=sample_n, replace=False)
X_imp_sample = X_test_scaled[sample_idx].copy()
y_imp_sample = y_test_int[sample_idx].copy()

# Permutation importance is computed only over the 20 selected feature columns.
base_probs = best_model.predict(X_imp_sample, verbose=0)
base_acc = accuracy_score(y_imp_sample, np.argmax(base_probs, axis=1))
importance_rows = []
feature_to_col = {f: i for i, f in enumerate(feature_names)}
for feat in importance_features:
    col = feature_to_col[feat]
    X_perm = X_imp_sample.copy()
    X_perm[:, col] = rng.permutation(X_perm[:, col])
    perm_pred = np.argmax(best_model.predict(X_perm, verbose=0), axis=1)
    importance_rows.append((feat, base_acc - accuracy_score(y_imp_sample, perm_pred)))

importance_df = pd.DataFrame(importance_rows, columns=["feature", "accuracy_decrease"]).sort_values(
    "accuracy_decrease", ascending=False
)
display(importance_df.head(15))

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=importance_df.head(15), x="accuracy_decrease", y="feature", ax=ax)
ax.set_title("Figure 11. Permutation Importance for Selected Features")
ax.set_xlabel("Decrease in sample accuracy after shuffling")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "11_permutation_importance.png", dpi=160, bbox_inches="tight")
plt.show()

print("Interpretation: larger positive accuracy decreases suggest that the model relies more on that feature in this sample.")
print("Caution: correlated features can share or mask importance; the result is not causal evidence.")

## 10. Export results for the report and GitHub
This cell writes measured results to a JSON file and exports key tables. Copy actual values from these outputs into the report template. Figures are saved in the `figures/` directory.

In [ ]:
results = {
    "dataset": "UCI Human Activity Recognition Using Smartphones",
    "training_rows_original": int(len(X_train_original)),
    "test_rows": int(len(X_test)),
    "feature_count": int(X_train_original.shape[1]),
    "class_names": [str(x) for x in class_names],
    "selected_config": best_config,
    "best_validation_accuracy": float(max(history.history["val_accuracy"])),
    "test_accuracy": float(test_accuracy),
    "macro_ovr_roc_auc": float(macro_auc),
    "class_roc_auc": auc_by_class,
    "classification_report": report_dict,
    "missing_feature_cells_training_original": int(X_train_original.isna().sum().sum()),
    "duplicate_feature_rows_training_original": int(X_train_original.duplicated().sum())
}
with open("results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

tuning_df.to_csv("tuning_results.csv", index=False)
activity_summary.to_csv("activity_distribution.csv")
importance_df.to_csv("permutation_importance.csv", index=False)

print(json.dumps(results, indent=2))
print("\nFiles created:")
print("- results_summary.json")
print("- tuning_results.csv")
print("- activity_distribution.csv")
print("- permutation_importance.csv")
print("- figures/*.png")

## 11. Conclusion writing guide
After running the notebook, complete the following using the observed results:
1. State which activity classes were easiest and hardest to distinguish based on per-class F1 and the confusion matrix.
2. Describe the most visible feature-distribution patterns and the strongest correlations actually shown in the heatmap.
3. Compare training and validation curves; identify evidence of convergence or overfitting.
4. Report the measured test accuracy and macro one-vs-rest ROC-AUC from the output.
5. Discuss limitations: engineered rather than raw time-series inputs, subject/population coverage, feature correlation, and the fact that this model is not a medical or safety-critical device.
6. Future work: subject-wise validation, 1D-CNN/LSTM models on raw time windows, broader participant data, calibration, and deployment on a mobile/edge device.

Do not paste an unrun metric or invent chart findings.

## References (IEEE style)
[1] D. Anguita, A. Ghio, L. Oneto, X. Parra, and J. L. Reyes-Ortiz, “A Public Domain Dataset for Human Activity Recognition Using Smartphones,” in *Proc. ESANN*, 2013.

[2] D. Anguita, A. Ghio, L. Oneto, X. Parra, and J. L. Reyes-Ortiz, “Human Activity Recognition on Smartphones Using a Multiclass Hardware-Friendly Support Vector Machine,” in *Proc. International Workshop on Ambient Assisted Living*, 2012.

[3] UCI Machine Learning Repository, “Human Activity Recognition Using Smartphones,” University of California, Irvine. [Online]. Available: https://archive.ics.uci.edu/dataset/240/humanactivityrecognitionusingsmartphones

[4] F. Pedregosa et al., “Scikit-learn: Machine Learning in Python,” *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.

[5] TensorFlow, “Keras: The Python Deep Learning API,” [Online]. Available: https://keras.io/

[6] J. D. Hunter, “Matplotlib: A 2D Graphics Environment,” *Computing in Science & Engineering*, vol. 9, no. 3, pp. 90–95, 2007.